# Day 2 Session 4 -- Demos: Multi-Agent Workflows & Agentic Systems

This notebook contains the **instructor-led demos** for Session 4. Run each cell, observe the output, and follow along as the instructor explains each multi-agent pattern.

**Engineering context:** You are an engineer building multi-agent orchestration systems. Your users (consultants) work in engagement teams where a Partner supervises, Associates specialize by domain, and work flows across functional boundaries. Multi-agent architectures mirror these team dynamics -- supervisor routing, agent handoffs, parallel workstreams, and collaborative deliverable creation.

In [1]:
!pip install -q langchain langchain-openai langchain-core langgraph python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


## Setup

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Literal
import json
import os

print("All imports successful!")

All imports successful!


---
## Demo 1: Supervisor-Worker Pattern -- Partner Delegates to Associates

The supervisor pattern is the most common multi-agent architecture, and it directly mirrors an engagement team. A **Partner** (supervisor) receives the client request, decides which **Associate** (worker) should handle it based on the nature of the work, and routes accordingly. One Associate handles financial analysis, another develops strategic recommendations, and a third builds implementation roadmaps. The Partner reviews all outputs before they reach the client.

This is the **supervisor-worker pattern** -- a central routing agent delegates to specialized workers and reviews their deliverables.

In [3]:
# Demo 1 - Supervisor-Worker Pattern: Partner Delegates to Associates

class SupervisorState(TypedDict):
    task: str
    assigned_to: str
    worker_output: str
    final_response: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def supervisor_node(state: SupervisorState) -> dict:
    """Partner assesses the engagement request and assigns to the right Associate."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Partner managing an engagement team. "
            "Based on the client request, assign the work to one Associate: "
            "'financial_analyst' (for financial modeling, valuation, or quantitative analysis), "
            "'strategy_consultant' (for market assessment, competitive strategy, or growth recommendations), or "
            "'operations_expert' (for process optimization, implementation planning, or operational improvements). "
            "Respond with just the associate role name."
        )),
        HumanMessage(content=state["task"])
    ])
    assigned = response.content.strip().lower()
    print(f"  Partner assigned to: {assigned}")
    return {"assigned_to": assigned}

def financial_analyst_node(state: SupervisorState) -> dict:
    """Associate specializing in financial analysis and modeling."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate specializing in financial analysis. "
            "Provide structured, data-driven analysis with key metrics, financial implications, "
            "and quantitative insights. Use consulting frameworks where appropriate."
        )),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": f"[Financial Analysis Workstream] {response.content}"}

def strategy_consultant_node(state: SupervisorState) -> dict:
    """Associate specializing in strategy and market assessment."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate specializing in strategy. "
            "Provide market insights, competitive positioning analysis, and strategic recommendations. "
            "Structure your response with clear hypotheses and supporting evidence."
        )),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": f"[Strategy Workstream] {response.content}"}

def operations_expert_node(state: SupervisorState) -> dict:
    """Associate specializing in operations and implementation."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate specializing in operations and implementation. "
            "Provide actionable implementation roadmaps, process improvements, and operational recommendations. "
            "Include timelines, resource requirements, and key milestones."
        )),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": f"[Operations Workstream] {response.content}"}

def review_node(state: SupervisorState) -> dict:
    """Partner reviews the Associate's deliverable before client presentation."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Partner reviewing an Associate's deliverable. "
            "Assess the quality, add executive-level perspective, and finalize the recommendation. "
            "Ensure the output is client-ready with clear, actionable insights."
        )),
        HumanMessage(content=f"Client Request: {state['task']}\n\nAssociate Deliverable:\n{state['worker_output']}")
    ])
    return {"final_response": response.content}

def route_to_worker(state: SupervisorState) -> str:
    a = state["assigned_to"]
    if "financial" in a: return "financial_analyst"
    if "strategy" in a: return "strategy_consultant"
    return "operations_expert"

graph = StateGraph(SupervisorState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("financial_analyst", financial_analyst_node)
graph.add_node("strategy_consultant", strategy_consultant_node)
graph.add_node("operations_expert", operations_expert_node)
graph.add_node("review", review_node)

graph.set_entry_point("supervisor")
graph.add_conditional_edges("supervisor", route_to_worker, {
    "financial_analyst": "financial_analyst",
    "strategy_consultant": "strategy_consultant",
    "operations_expert": "operations_expert"
})
graph.add_edge("financial_analyst", "review")
graph.add_edge("strategy_consultant", "review")
graph.add_edge("operations_expert", "review")
graph.add_edge("review", END)

app = graph.compile()

for task in [
    "Assess the financial viability of acquiring a mid-size SaaS company with $50M ARR",
    "Develop a market entry strategy for a luxury automotive brand entering India",
    "Design a lean operating model for a post-merger integration of two regional banks"
]:
    print(f"\nClient Request: {task}")
    result = app.invoke({"task": task, "assigned_to": "", "worker_output": "", "final_response": ""})
    print(f"Partner-Reviewed Deliverable: {result['final_response'][:250]}...")


Client Request: Assess the financial viability of acquiring a mid-size SaaS company with $50M ARR
  Partner assigned to: financial_analyst
Partner-Reviewed Deliverable: ### Executive Assessment of Associate's Deliverable

The deliverable provides a structured and comprehensive analysis of the financial viability of acquiring a mid-size SaaS company with $50 million in ARR. The framework is well-organized, covering e...

Client Request: Develop a market entry strategy for a luxury automotive brand entering India
  Partner assigned to: strategy_consultant
Partner-Reviewed Deliverable: ### Assessment of Associate Deliverable

The deliverable presents a well-structured market entry strategy for a luxury automotive brand entering India. The analysis is comprehensive, covering key market insights, competitive positioning, and actionab...

Client Request: Design a lean operating model for a post-merger integration of two regional banks
  Partner assigned to: operations_expert
Partner-Reviewe

---
## Demo 2: Agent Handoff -- Strategy to Operations to Implementation

In engagements, work often flows between specialized teams: the **Strategy team** defines the vision and hypotheses, the **Operations team** translates strategy into operational plans, and the **Implementation team** builds the execution roadmap. Each handoff includes structured context -- engagement scope, key findings, and areas requiring attention.

This is the **agent handoff pattern** where one agent transfers control and context to the next agent in a pipeline.

In [4]:
# Demo 2 - Agent Handoff: Strategy -> Operations -> Implementation

class HandoffState(TypedDict):
    user_request: str
    draft: str
    review_notes: str
    final_output: str
    handoff_context: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

def strategy_team_agent(state: HandoffState) -> dict:
    """Strategy team: defines the strategic vision, hypotheses, and recommendations."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Strategy team. Analyze the client situation and develop "
            "strategic recommendations. Structure your output with: key hypotheses, "
            "supporting evidence, and top-level strategic recommendations."
        )),
        HumanMessage(content=state["user_request"])
    ])
    context = (
        "Strategy phase complete. Key deliverables: strategic hypotheses validated, "
        "market positioning defined, growth levers identified. "
        "Operations team should focus on translating these into operational plans."
    )
    print(f"  [Strategy Team] Delivered strategic recommendations ({len(response.content)} chars)")
    return {"draft": response.content, "handoff_context": context}

def operations_team_agent(state: HandoffState) -> dict:
    """Operations team: translates strategy into operational plans and processes."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Operations team. Review the strategy deliverable and "
            "develop an operational plan. Focus on process changes, resource allocation, "
            "organizational structure, and key performance indicators."
        )),
        HumanMessage(content=f"Handoff Context: {state['handoff_context']}\n\nStrategy Deliverable:\n{state['draft']}")
    ])
    print(f"  [Operations Team] Operational plan ready ({response.content[:80]}...)")
    return {"review_notes": response.content}

def implementation_team_agent(state: HandoffState) -> dict:
    """Implementation team: builds the execution roadmap with timelines and milestones."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Implementation team. Take the strategy and operational plans "
            "and create a concrete implementation roadmap. Include: phased timeline, "
            "key milestones, resource requirements, risk mitigation, and success metrics."
        )),
        HumanMessage(content=f"Strategy:\n{state['draft']}\n\nOperational Plan:\n{state['review_notes']}")
    ])
    print(f"  [Implementation Team] Roadmap finalized ({len(response.content)} chars)")
    return {"final_output": response.content}

graph = StateGraph(HandoffState)
graph.add_node("strategy", strategy_team_agent)
graph.add_node("operations", operations_team_agent)
graph.add_node("implementation", implementation_team_agent)
graph.set_entry_point("strategy")
graph.add_edge("strategy", "operations")
graph.add_edge("operations", "implementation")
graph.add_edge("implementation", END)
app = graph.compile()

result = app.invoke({
    "user_request": "Our client, a Fortune 500 retailer, needs to develop a digital transformation strategy to compete with e-commerce disruptors",
    "draft": "", "review_notes": "", "final_output": "", "handoff_context": ""
})
print(f"\nImplementation Roadmap:\n{result['final_output'][:400]}...")

  [Strategy Team] Delivered strategic recommendations (4210 chars)
  [Operations Team] Operational plan ready (### Operational Plan

To translate the strategic recommendations into actionable...)
  [Implementation Team] Roadmap finalized (5397 chars)

Implementation Roadmap:
## Implementation Roadmap

### Overview
This implementation roadmap outlines a phased approach to executing the strategic and operational plans. It includes a timeline, key milestones, resource requirements, risk mitigation strategies, and success metrics.

### Phased Timeline

| Phase | Duration | Key Activities | Key Milestones |
|-------|----------|----------------|-----------------|
| **Phase ...


---
## Demo 3: Parallel Execution (Fan-out/Fan-in) -- M&A Due Diligence Workstreams

In an M&A due diligence engagement, multiple workstreams run simultaneously: **financial analysis**, **market research**, and **competitive assessment** all proceed in parallel to meet tight deal timelines. Each workstream produces independent findings, and the Engagement Manager synthesizes them into a unified due diligence report.

This is the **parallel execution pattern** -- independent agents work on different aspects, and a merger step combines the results.

In [5]:
# Demo 3 - Parallel Agent Execution: M&A Due Diligence Workstreams (Simulated)

class ParallelState(TypedDict):
    topic: str
    financial_analysis: str
    market_research: str
    competitive_assessment: str
    merged_report: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

def financial_analysis_agent(state: ParallelState) -> dict:
    """Workstream 1: Financial due diligence -- revenue, margins, projections."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey financial analyst conducting M&A due diligence. "
            "Analyze the financial aspects: revenue trends, margin structure, "
            "cash flow quality, and valuation considerations. Provide 3 key findings."
        )),
        HumanMessage(content=state["topic"])
    ])
    print("  [Financial Analysis Workstream] Complete")
    return {"financial_analysis": r.content}

def market_research_agent(state: ParallelState) -> dict:
    """Workstream 2: Market due diligence -- TAM, growth drivers, trends."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey market research analyst conducting M&A due diligence. "
            "Assess the target market: total addressable market (TAM), growth drivers, "
            "regulatory landscape, and market trends. Provide 3 key findings."
        )),
        HumanMessage(content=state["topic"])
    ])
    print("  [Market Research Workstream] Complete")
    return {"market_research": r.content}

def competitive_assessment_agent(state: ParallelState) -> dict:
    """Workstream 3: Competitive due diligence -- positioning, moats, threats."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey competitive strategy analyst conducting M&A due diligence. "
            "Evaluate competitive dynamics: market positioning, competitive moats, "
            "threat of new entrants, and differentiation. Provide 3 key findings."
        )),
        HumanMessage(content=state["topic"])
    ])
    print("  [Competitive Assessment Workstream] Complete")
    return {"competitive_assessment": r.content}

def synthesis_agent(state: ParallelState) -> dict:
    """Engagement Manager synthesizes all workstreams into a unified due diligence report."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Engagement Manager synthesizing M&A due diligence findings. "
            "Combine the three workstream outputs into a cohesive executive summary. "
            "Include: overall recommendation (proceed/caution/pass), key risks, "
            "synergy opportunities, and critical next steps."
        )),
        HumanMessage(content=(
            f"Deal Context: {state['topic']}\n\n"
            f"Financial Analysis:\n{state['financial_analysis']}\n\n"
            f"Market Research:\n{state['market_research']}\n\n"
            f"Competitive Assessment:\n{state['competitive_assessment']}"
        ))
    ])
    return {"merged_report": r.content}

graph = StateGraph(ParallelState)
graph.add_node("financial", financial_analysis_agent)
graph.add_node("market", market_research_agent)
graph.add_node("competitive", competitive_assessment_agent)
graph.add_node("synthesis", synthesis_agent)

graph.set_entry_point("financial")
graph.add_edge("financial", "market")
graph.add_edge("market", "competitive")
graph.add_edge("competitive", "synthesis")
graph.add_edge("synthesis", END)

app = graph.compile()

result = app.invoke({
    "topic": "Acquisition of a mid-market healthcare technology company specializing in AI-powered diagnostics, with $80M revenue and 25% YoY growth",
    "financial_analysis": "", "market_research": "", "competitive_assessment": "", "merged_report": ""
})
print(f"\nDue Diligence Report:\n{result['merged_report'][:400]}...")

  [Financial Analysis Workstream] Complete
  [Market Research Workstream] Complete
  [Competitive Assessment Workstream] Complete

Due Diligence Report:
### Executive Summary: M&A Due Diligence Findings for Acquisition of AI-Powered Diagnostics Company

#### Overall Recommendation: Proceed with Caution

The acquisition of the mid-market healthcare technology company specializing in AI-powered diagnostics presents a compelling opportunity, given its strong revenue growth and favorable market conditions. However, careful consideration of key risks a...


---
## Demo 4: Collaborative Deliverable Creation -- Engagement Team Client Presentation

Producing a client deliverable is a collaborative effort. The **Engagement Manager** creates the storyline and structure (using the pyramid principle), an **Associate** develops the detailed content and analyses, and a **Senior Partner** reviews and polishes the final presentation for the Steering Committee.

This is the **collaborative writing pattern** where specialized agents each contribute to building a shared output sequentially.

In [6]:
# Demo 4 - Collaborative Deliverable Creation: Engagement Team Client Presentation

class CollabState(TypedDict):
    topic: str
    outline: str
    draft: str
    polished: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")))

def em_storyline(state: CollabState) -> dict:
    """Engagement Manager creates the presentation storyline and structure."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Engagement Manager creating a presentation storyline. "
            "Develop a clear 3-section structure with the pyramid principle: "
            "start with the answer, then supporting arguments, then evidence. "
            "Include section titles and key messages for each section."
        )),
        HumanMessage(content=f"Create a client presentation storyline for: {state['topic']}")
    ])
    print("  [EM] Storyline created")
    return {"outline": r.content}

def associate_content(state: CollabState) -> dict:
    """Associate develops detailed content following the storyline."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate developing content for a client presentation. "
            "Follow the storyline structure and fill in detailed analysis, data points, "
            "and recommendations. Use clear, concise consulting language. "
            "Keep the content executive-ready."
        )),
        HumanMessage(content=f"Engagement Topic: {state['topic']}\n\nStoryline:\n{state['outline']}")
    ])
    print(f"  [Associate] Content developed ({len(r.content)} chars)")
    return {"draft": r.content}

def partner_review(state: CollabState) -> dict:
    """Senior Partner polishes the presentation for Steering Committee readiness."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Senior Partner reviewing a client presentation. "
            "Polish the content for C-suite readability: sharpen the 'so what', "
            "ensure each section drives toward actionable recommendations, "
            "and add strategic perspective. Make it Steering Committee-ready."
        )),
        HumanMessage(content=state["draft"])
    ])
    print(f"  [Partner] Presentation polished ({len(r.content)} chars)")
    return {"polished": r.content}

graph = StateGraph(CollabState)
graph.add_node("em_storyline", em_storyline)
graph.add_node("associate_content", associate_content)
graph.add_node("partner_review", partner_review)
graph.set_entry_point("em_storyline")
graph.add_edge("em_storyline", "associate_content")
graph.add_edge("associate_content", "partner_review")
graph.add_edge("partner_review", END)
app = graph.compile()

result = app.invoke({
    "topic": "Cross-functional transformation program for a global pharmaceutical company entering biosimilars",
    "outline": "", "draft": "", "polished": ""
})
print(f"\nSteering Committee Presentation:\n{result['polished'][:500]}...")

  [EM] Storyline created
  [Associate] Content developed (4995 chars)
  [Partner] Presentation polished (5215 chars)

Steering Committee Presentation:
### Presentation Title: Strategic Cross-Functional Transformation for Biosimilars Market Entry

---

#### Section 1: Strategic Imperative for Entering Biosimilars
**Key Message:** Entering the biosimilars market is not just an opportunity; it is a strategic necessity for sustainable growth and competitive differentiation in the pharmaceutical sector.

- **Market Growth Potential:** The global biosimilars market is on a trajectory to reach approximately $50 billion by 2025, with a remarkable CAGR...


---
## Demo 5: End-to-End Multi-Agent System -- Engagement Intake & Delivery with Intent Classification

This demo brings together supervisor routing, specialized workers, and quality review into a complete engagement delivery system. A **Triage Partner** classifies incoming client requests by type (strategic question, execution task, or relationship touchpoint), routes to the appropriate specialist, and a **Quality Assurance** step ensures deliverable standards before client delivery.

This mirrors how diverse client interactions are managed through a structured intake and delivery process.

In [7]:
# Demo 5 - End-to-End: Engagement Intake & Delivery System

class E2EState(TypedDict):
    user_input: str
    intent: str
    agent_output: str
    quality_score: str
    final_output: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def intent_classifier(state: E2EState) -> dict:
    """Triage Partner classifies the client request type."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Partner triaging client requests. "
            "Classify the request as: 'strategic_question' (needs analysis or recommendation), "
            "'execution_task' (needs an action plan or deliverable), or "
            "'relationship' (general check-in or discussion). "
            "Respond with one classification only."
        )),
        HumanMessage(content=state["user_input"])
    ])
    intent = r.content.strip().lower()
    print(f"  Triage: {intent}")
    return {"intent": intent}

def strategy_advisor(state: E2EState) -> dict:
    """Handles strategic questions with hypothesis-driven analysis."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey strategy advisor. Answer the client's strategic question "
            "with structured, hypothesis-driven analysis. Use frameworks like Porter's Five Forces, "
            "value chain analysis, or growth-share matrix as appropriate."
        )),
        HumanMessage(content=state["user_input"])
    ])
    return {"agent_output": r.content}

def execution_lead(state: E2EState) -> dict:
    """Handles execution tasks with actionable implementation plans."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey execution lead. Develop a step-by-step action plan "
            "for the client's request. Include timeline, resource requirements, "
            "key milestones, and risk factors."
        )),
        HumanMessage(content=state["user_input"])
    ])
    return {"agent_output": r.content}

def relationship_manager(state: E2EState) -> dict:
    """Handles relationship touchpoints with a client-service mindset."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey relationship manager. Respond to the client with warmth "
            "and professionalism. Acknowledge their situation, provide value-adding insights, "
            "and suggest next steps to deepen the engagement."
        )),
        HumanMessage(content=state["user_input"])
    ])
    return {"agent_output": r.content}

def quality_check(state: E2EState) -> dict:
    """QA check ensures the deliverable meets standards."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey quality assurance reviewer. Rate this deliverable 1-10 "
            "on: clarity, actionability, and strategic depth. "
            'Output JSON: {"score": N, "feedback": "..."}'
        )),
        HumanMessage(content=f"Client Request: {state['user_input']}\n\nDeliverable: {state['agent_output']}")
    ])
    return {"quality_score": r.content, "final_output": state["agent_output"]}

def route_intent(state: E2EState) -> str:
    if "strategic" in state["intent"] or "question" in state["intent"]: return "strategy"
    if "execution" in state["intent"] or "task" in state["intent"]: return "execution"
    return "relationship"

graph = StateGraph(E2EState)
graph.add_node("classify", intent_classifier)
graph.add_node("strategy", strategy_advisor)
graph.add_node("execution", execution_lead)
graph.add_node("relationship", relationship_manager)
graph.add_node("quality", quality_check)

graph.set_entry_point("classify")
graph.add_conditional_edges("classify", route_intent, {
    "strategy": "strategy", "execution": "execution", "relationship": "relationship"
})
graph.add_edge("strategy", "quality")
graph.add_edge("execution", "quality")
graph.add_edge("relationship", "quality")
graph.add_edge("quality", END)
app = graph.compile()

for inp in [
    "What are the key factors driving consolidation in the European banking sector?",
    "Develop a 90-day plan to reduce supply chain costs by 15%",
    "We enjoyed the last workshop -- can we schedule a follow-up to discuss next steps?"
]:
    print(f"\nClient: {inp}")
    r = app.invoke({"user_input": inp, "intent": "", "agent_output": "", "quality_score": "", "final_output": ""})
    print(f"Deliverable: {r['final_output'][:200]}...")
    print(f"QA Score: {r['quality_score'][:100]}")


Client: What are the key factors driving consolidation in the European banking sector?
  Triage: strategic_question
Deliverable: To analyze the key factors driving consolidation in the European banking sector, we can employ a structured, hypothesis-driven approach using frameworks such as Porter's Five Forces and the value chai...
QA Score: {"score": 8, "feedback": "The deliverable provides a clear and structured analysis of the key factor

Client: Develop a 90-day plan to reduce supply chain costs by 15%
  Triage: execution_task
Deliverable: ### 90-Day Action Plan to Reduce Supply Chain Costs by 15%

#### Objective:
To achieve a 15% reduction in supply chain costs within 90 days through strategic analysis, process optimization, and stakeh...
QA Score: {
  "score": 8,
  "feedback": "The deliverable is clear and well-structured, outlining a comprehensi

Client: We enjoyed the last workshop -- can we schedule a follow-up to discuss next steps?
  Triage: relationship
Deliverable: Dear [C

---
## Summary

This demo notebook covered five multi-agent patterns:

1. **Supervisor-Worker** -- A central supervisor routes tasks to specialized worker agents based on task type
2. **Agent Handoff** -- Agents pass structured context to each other, enabling sequential specialization
3. **Parallel Execution (Fan-out/Fan-in)** -- Independent subtasks run concurrently, with a merger step combining results
4. **Collaborative Deliverable** -- Multiple agents contribute different aspects to a shared output
5. **End-to-End with Intent Classification** -- Complete architectures combining classification, specialized agents, and quality review